GPE 

In [ ]:
import h5py
import matplotlib.pyplot as plt
import cv2
import numpy as np

def load_events(file_path):
    with h5py.File(file_path, 'r') as f:
        e = f['CD/events'][:]
        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }

def select_large_roi(data, x_min, x_max, y_min, y_max, t_min, t_max):
    mask = (data['x'] >= x_min) & (data['x'] <= x_max) & \
           (data['y'] >= y_min) & (data['y'] <= y_max) & \
           (data['t'] >= t_min) & (data['t'] <= t_max)
    return {k: v[mask] for k, v in data.items()}


def events_to_frames(events, frame_dt=0.002):

    xmin, xmax = int(events['x'].min()), int(events['x'].max())
    ymin, ymax = int(events['y'].min()), int(events['y'].max())

    W = xmax - xmin + 1
    H = ymax - ymin + 1

    t0, t1 = events['t'].min(), events['t'].max()

    frames = []
    times = []

    for k in range(int((t1 - t0) / frame_dt)):

        ta = t0 + k * frame_dt
        tb = ta + frame_dt

        mask = (events['t'] >= ta) & (events['t'] < tb)

        if np.sum(mask) < 50:
            continue

        frame = np.zeros((H, W), np.float32)

        xs = (events['x'][mask] - xmin).astype(int)
        ys = (events['y'][mask] - ymin).astype(int)

        for x, y in zip(xs, ys):
            frame[y, x] += 1

        frames.append(frame)
        times.append((ta + tb) / 2)

    return np.array(frames), np.array(times)


def complex_gabor(frame, ksize=31, sigma=4.0, theta=np.pi/2, lambd=10.0):

    # 实部
    real = cv2.getGaborKernel((ksize, ksize), sigma, theta, lambd, gamma=0.5, psi=0)

    # 虚部（90°相移）
    imag = cv2.getGaborKernel((ksize, ksize), sigma, theta, lambd, gamma=0.5, psi=np.pi/2)

    real_resp = cv2.filter2D(frame, cv2.CV_32F, real)
    imag_resp = cv2.filter2D(frame, cv2.CV_32F, imag)

    return real_resp + 1j * imag_resp
def method_phase_csp(frames, times):

    complex_stack = []

    for f in frames:

        # ⭐ 关键：选方向（垂直方向，因为是上下振动）
        resp = complex_gabor(f, theta=np.pi/2)

        complex_stack.append(resp)

    complex_stack = np.array(complex_stack)

    # 相位 & 幅值
    phase = np.angle(complex_stack)
    amp   = np.abs(complex_stack)

    # ===== 时间展开（关键）=====
    phase_unwrap = np.unwrap(phase, axis=0)

    # ===== 相位差 =====
    dphi = np.diff(phase_unwrap, axis=0)

    dt = times[1] - times[0]

    # ===== 加权平均（抗噪关键）=====
    signal = np.sum(dphi * amp[:-1], axis=(1,2)) / \
             (np.sum(amp[:-1], axis=(1,2)) + 1e-6)

    # ===== 频率估计 =====
    fft = np.fft.rfft(signal)
    freqs = np.fft.rfftfreq(len(signal), dt)

    f_est = freqs[np.argmax(np.abs(fft))]

    return f_est, signal

def main():

    # file_path = "./data/4.14_20hz.hdf5"
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     600, 1000,
    #     100, 180,
    #     0, 0.2
    # )

    # file_path = './data/20hz-2-20.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     100, 300,
    #     420, 470,
    #     0, 0.8
    # )


    # file_path = './data/zdbiaochi.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     500, 800,
    #     400, 600,
    #     0, 0.2
    # )

    file_path = "./data/30hz-20.hdf5"
    data = load_events(file_path)
    roi = select_large_roi(
        data,
        100, 600,
        420, 460,
        0, 1.0
    )    
    
    # file_path = './data/200hz-20.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     200, 600,
    #     350, 460,
    #     0, 0.5
    # )
    
    frames, times = events_to_frames(roi, frame_dt=0.001)

    print(f"Frames: {len(frames)}")

    f_est, signal = method_phase_csp(frames, times)

    print(f"Estimated Frequency: {f_est:.2f} Hz")

    # 可视化信号
    plt.plot(signal)
    plt.title("Phase-based vibration signal")
    plt.show()


if __name__ == "__main__":
    main()

ECT 

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d

# =========================================================
# 1. 读取事件数据
# =========================================================
def load_events(file_path):
    with h5py.File(file_path, 'r') as f:
        if 'CD' in f and 'events' in f['CD']:
            e = f['CD/events'][:]

            return {
                'x': e['x'].astype(float),
                'y': e['y'].astype(float),
                't': e['t'].astype(float) * 1e-6,   # us → s
                'p': e['p'].astype(float)
            }

    raise ValueError("无效的 HDF5 文件结构")


# =========================================================
# 2. ROI + 时间截取
# =========================================================
def select_roi(events, roi, t_range):

    x, y, t = events['x'], events['y'], events['t']

    rx0, rx1, ry0, ry1 = roi
    t0, t1 = t_range

    m = (
        (x >= rx0) & (x <= rx1) &
        (y >= ry0) & (y <= ry1) &
        (t >= t0) & (t <= t1)
    )

    return {
        'x': x[m],
        'y': y[m],
        't': t[m],
        'p': events['p'][m]
    }


# =========================================================
# 3. ECT：事件质心轨迹
# =========================================================
def event_centroid_tracking(events,
                             dt=0.001,
                             min_events=30,
                             smooth_sigma=1.0):

    t = events['t']
    y = events['y']

    t0 = t.min()
    t1 = t.max()

    centers = []
    times = []

    n_bins = int((t1 - t0) // dt)

    for i in range(n_bins):

        ta = t0 + i * dt
        tb = ta + dt

        m = (t >= ta) & (t < tb)

        # 事件太少则跳过
        if np.sum(m) < min_events:
            continue

        # =================================================
        # ECT核心：
        # 直接对事件坐标求质心
        # =================================================
        yc = np.mean(y[m])

        centers.append(yc)
        times.append((ta + tb) / 2)

    centers = np.array(centers)
    times = np.array(times)

    # 去 DC
    centers -= np.mean(centers)

    # 平滑
    centers = gaussian_filter1d(centers, sigma=smooth_sigma)

    return times, centers


# =========================================================
# 4. Phase + FFT 振动分析
# =========================================================
def phase_with_fft(times,
                   signal,
                   show_spectrum=True,
                   save_path="ect_fft.png"):

    # -----------------------------------------------------
    # 参数
    # -----------------------------------------------------
    grid_alpha = 0.3
    label_fontsize = 20
    tick_fontsize = 18

    # -----------------------------------------------------
    # 去均值
    # -----------------------------------------------------
    signal = signal - np.mean(signal)

    # =====================================================
    # Hilbert 相位分析（辅助）
    # =====================================================
    analytic = hilbert(signal)

    phase = np.unwrap(np.angle(analytic))

    inst_freq = np.diff(phase) / (
        2 * np.pi * np.diff(times)
    )

    f_phase = np.median(inst_freq)

    # =====================================================
    # FFT 主频分析（最终结果）
    # =====================================================
    fs = 1.0 / np.mean(np.diff(times))

    N = len(signal)

    freqs = np.fft.rfftfreq(N, d=1/fs)

    spectrum = np.abs(np.fft.rfft(signal))

    spectrum /= spectrum.max() + 1e-12

    idx = np.argmax(spectrum)

    f_fft = freqs[idx]

    amp_fft = spectrum[idx]

    # =====================================================
    # 可视化频谱
    # =====================================================
    if show_spectrum:

        plt.figure(figsize=(6, 5))

        plt.plot(freqs, spectrum, lw=2)

        plt.scatter(
            f_fft,
            amp_fft,
            color='r',
            s=60,
            zorder=3,
            label=f"{f_fft:.2f} Hz"
        )

        plt.xlabel(
            "Frequency (Hz)",
            fontsize=label_fontsize
        )

        plt.ylabel(
            "Normalized Amplitude",
            fontsize=label_fontsize
        )

        plt.xlim(0, 250)

        plt.legend(
            frameon=False,
            fontsize=tick_fontsize
        )

        plt.grid(alpha=grid_alpha)

        plt.tick_params(
            axis='both',
            labelsize=tick_fontsize
        )

        plt.tight_layout()

        # plt.savefig(save_path,
        #             dpi=300,
        #             bbox_inches='tight')

        plt.show()

    return f_phase, f_fft, signal


# =========================================================
# 5. 可视化 ECT 轨迹
# =========================================================
def plot_centroid_signal(times, signal):

    plt.figure(figsize=(6, 5))

    plt.plot(times, signal, lw=1.5)

    plt.xlabel("Time (s)", fontsize=16)

    plt.ylabel("Centroid Position (px)", fontsize=16)

    plt.title("ECT Vibration Signal", fontsize=18)

    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.show()


# =========================================================
# 6. 主程序
# =========================================================
if __name__ == "__main__":

    # =====================================================
    # 用户设置
    # =====================================================

    FILE_PATH = "./data/30hz-20.hdf5"
    ROI = (100, 600, 420, 460)

    # FILE_PATH = "./data/zdbiaochi.hdf5"
    # ROI = (600, 700, 400, 600)

    # FILE_PATH = './data/100hz-20.hdf5'  # 替换为你的HDF5文件路径
    # ROI = (200, 600, 350, 460)

    # FILE_PATH = './data/200hz-20.hdf5'  # 替换为你的HDF5文件路径
    # ROI = (200, 600, 350, 460)
    
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     200, 600,
    #     350, 460,
    #     0, 0.5
    # )
    
    T_RANGE = (0.0, 1.0)

    # -----------------------------------------------------
    # ECT 时间窗口
    # -----------------------------------------------------
    ECT_DT = 0.001

    # =====================================================
    # 读取事件
    # =====================================================
    events = load_events(FILE_PATH)

    # =====================================================
    # ROI 截取
    # =====================================================
    events = select_roi(
        events,
        ROI,
        T_RANGE
    )

    print("Events after ROI:", len(events['t']))

    # =====================================================
    # ECT 质心轨迹
    # =====================================================
    t_c, centers = event_centroid_tracking(
        events,
        dt=ECT_DT,
        min_events=30,
        smooth_sigma=1.0
    )

    # =====================================================
    # 可视化振动轨迹
    # =====================================================
    plot_centroid_signal(t_c, centers)

    # =====================================================
    # Phase + FFT
    # =====================================================
    f_phase, f_fft, y_signal = phase_with_fft(
        t_c,
        centers
    )

    # =====================================================
    # 输出结果
    # =====================================================
    amplitude = 0.5 * (
        y_signal.max() - y_signal.min()
    )

    print("\n========== ECT Vibration Result ==========")

    print(
        f"Auxiliary Phase Frequency : "
        f"{f_phase:.3f} Hz"
    )

    print(
        f"Final FFT Dominant Frequency : "
        f"{f_fft:.3f} Hz"
    )

    print(
        f"Estimated Amplitude : "
        f"{amplitude:.3f} px"
    )

3DFIT

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# =========================================================
# 1. 数据读取与 ROI
# =========================================================
def load_events(file_path):
    with h5py.File(file_path, 'r') as f:
        e = f['CD/events'][:]
        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }

def select_large_roi(data, x_min, x_max, y_min, y_max, t_min, t_max):
    mask = (data['x'] >= x_min) & (data['x'] <= x_max) & \
           (data['y'] >= y_min) & (data['y'] <= y_max) & \
           (data['t'] >= t_min) & (data['t'] <= t_max)
    return {k: v[mask] for k, v in data.items()}

# =========================================================
# 2. 空间分组
# =========================================================
def group_by_x(roi_data, group_size=200):
    x_min, x_max = roi_data['x'].min(), roi_data['x'].max()
    bounds = np.arange(x_min, x_max + group_size, group_size)
    groups = []
    for i in range(len(bounds) - 1):
        x0, x1 = bounds[i], bounds[i + 1]
        mask = (roi_data['x'] >= x0) & (roi_data['x'] < x1)
        if np.sum(mask) < 100: continue
        groups.append({'id': i, 'x_range': (x0, x1), 'data': {k: v[mask] for k, v in roi_data.items()}})
    return groups

# =========================================================
# 3. 隐式模型与拟合
# =========================================================
def fit_group_implicit(group):
    data = group['data']
    x_raw, y_raw, t_raw = data['x'], data['y'], data['t']
    
    # --- 数值稳定性：去中心化 ---
    x_m, y_m = np.mean(x_raw), np.mean(y_raw)
    t_0 = t_raw.min()
    x, y, t = x_raw - x_m, y_raw - y_m, t_raw - t_0

    # 初始频率估计 (利用 y 轴方向波动作初值)
    y_c = y - np.mean(y)
    freqs = np.linspace(1, 300, 200)
    power = [np.abs(np.sum(y_c * np.exp(-1j*2*np.pi*f*t))) for f in freqs]
    f_init = freqs[np.argmax(power)]

    # 参数：[A, B, C, f, phi, lamb, D]
    # 隐式方程：Ax + By + C*exp(-lamb*t)*sin(2*pi*f*t + phi) + D = 0
    p0 = [0.0, 1.0, np.ptp(y)/2, f_init, 0.0, 0.1, 0.0]

    def objective(p):
        A, B, C, f, phi, lamb, D = p
        # 几何约束：A^2 + B^2 = 1 保证残差即为法向距离
        constraint_penalty = 1e6 * (np.sqrt(A**2 + B**2) - 1)**2
        
        vibration = C * np.exp(-lamb * t) * np.sin(2 * np.pi * f * t + phi)
        residual = A * x + B * y + vibration + D
        return np.mean(residual**2) + constraint_penalty

    res = minimize(objective, p0, method='L-BFGS-B', 
                   bounds=[(-1,1), (-1,1), (0,None), (1,500), (-np.pi, np.pi), (0,10), (None,None)])
    
    if not res.success: return None
    
    # 归一化结果
    popt = res.x
    norm = np.sqrt(popt[0]**2 + popt[1]**2)
    popt[[0,1,2,6]] /= norm

    return {
        'params': popt, 'x_m': x_m, 'y_m': y_m, 't_0': t_0, 'id': group['id'], 'x_range': group['x_range']
    }

# =========================================================
# 4. 可视化
# =========================================================
def plot_raw_and_surface(roi_data, fitted_groups):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # --- 1. 先可视化原始散点数据 ---
    # 使用较小的 s (点大小) 和 alpha (透明度) 以免遮挡曲面
    ax.scatter(
        roi_data['x'], 
        roi_data['t'], 
        roi_data['y'], 
        s=1, 
        alpha=0.15, 
        c='dodgerblue', 
        label='Raw Events'
    )

    # --- 2. 再可视化拟合的曲面 ---
    for g in fitted_groups:
        p = g['params']
        A, B, C, f, phi, lamb, D = p
        x0, x1 = g['x_range']
        
        # 生成网格数据（用于绘制曲面）
        xg = np.linspace(x0, x1, 50)
        tg = np.linspace(roi_data['t'].min(), roi_data['t'].max(), 100)
        XG, TG = np.meshgrid(xg, tg)
        
        # 局部坐标转换
        xl = XG - g['x_m']
        tl = TG - g['t_0']
        
        # 计算隐式曲面的 Y 值 (基于去中心化参数)
        vibration = C * np.exp(-lamb * tl) * np.sin(2 * np.pi * f * tl + phi)
        # yl = -(Ax + V + D) / B
        yl = -(A * xl + vibration + D) / B
        
        # 还原回全局 Y 坐标
        YG = yl + g['y_m']
        
        # 绘制曲面，使用 alpha=0.6 保持半透明，以便观察内部点云
        surf = ax.plot_surface(
            XG, TG, YG, 
            cmap='viridis', 
            alpha=0.6, 
            linewidth=0, 
            antialiased=True
        )

    # 设置标签和视角
    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("3DFIT: Raw Events vs Fitted Damped Surface")
    
    # 调整视角以便更好地观察拟合贴合度
    ax.view_init(elev=20, azim=-35) 
    
    plt.tight_layout()
    plt.show()
def plot_raw_events_only(roi_data):
    """图 1：仅展示原始事件散点图"""
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(
        roi_data['x'], 
        roi_data['t'], 
        roi_data['y'], 
        s=1, 
        alpha=0.3, 
        c='dodgerblue', 
        edgecolors='none'
    )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("Figure 1: Raw Event Spatiotemporal Cloud")
    ax.view_init(elev=20, azim=-35)
    plt.tight_layout()
    plt.show()

def plot_fitted_surface_only(roi_data, fitted_groups):
    """图 2：仅展示拟合后的隐式阻尼曲面"""
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    for g in fitted_groups:
        p = g['params']
        A, B, C, f, phi, lamb, D = p
        x0, x1 = g['x_range']
        
        # 生成网格
        xg = np.linspace(x0, x1, 60)
        tg = np.linspace(roi_data['t'].min(), roi_data['t'].max(), 120)
        XG, TG = np.meshgrid(xg, tg)
        
        # 局部坐标计算
        xl = XG - g['x_m']
        tl = TG - g['t_0']
        
        # 隐式转显式计算 YG
        vibration = C * np.exp(-lamb * tl) * np.sin(2 * np.pi * f * tl + phi)
        yl = -(A * xl + vibration + D) / B
        YG = yl + g['y_m']
        
        # 绘制曲面
        surf = ax.plot_surface(
            XG, TG, YG, 
            cmap='viridis', 
            alpha=0.9, 
            antialiased=True,
            rcount=100, ccount=100
        )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("Figure 2: Fitted Implicit Damped Surface")
    ax.view_init(elev=20, azim=-35)
    plt.tight_layout()
    plt.show()
# =========================================================
# 5. Main
# =========================================================
def main():
    # file_path = "./data/zdbiaochi.hdf5" # 替换为你的路径
    # # data = load_events(file_path)
    # # roi = select_large_roi(data, 300, 1000, 400, 600, 0, 0.2)



    
    # file_path = "./data/zdbiaochi.hdf5" # 替换为你的路径
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     300, 1000,
    #     400, 600,
    #     0, 0.2
    # )
    
    
    # file_path = './data/200hz-20.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)

    # roi = select_large_roi(
    #     data,
    #     200, 600,
    #     350, 460,
    #     0, 0.5
    # )


    file_path = "./data/30hz-20.hdf5"
    data = load_events(file_path)
    roi = select_large_roi(
        data,
        100, 600,
        420, 460,
        0, 1.0
    )    
    
    
    print(f"Events in ROI: {len(roi['x'])}")
    groups = group_by_x(roi, group_size=800)
    
    fitted = []
    for g in groups:
        res = fit_group_implicit(g)
        if res:
            fitted.append(res)
            print(f"Group {res['id']} Fitted: f={res['params'][3]:.2f}Hz, λ={res['params'][5]:.2f}")

    # if fitted:
    #     plot_raw_and_surface(roi, fitted)


# 假设得到的拟合结果存储在 fitted 列表中
    if not fitted:
        print("拟合失败，无数据可绘。")
        return

    # --- 弹出图 1：原始数据 ---
    print("正在生成图 1：原始散点...")
    plot_raw_events_only(roi)

    # --- 弹出图 2：拟合曲面 ---
    print("正在生成图 2：拟合曲面...")
    plot_fitted_surface_only(roi, fitted)
if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# =========================================================
# 1. 数据读取与 ROI
# =========================================================
def load_events(file_path):

    with h5py.File(file_path, 'r') as f:

        e = f['CD/events'][:]

        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }


def select_large_roi(
        data,
        x_min, x_max,
        y_min, y_max,
        t_min, t_max):

    mask = (
        (data['x'] >= x_min) &
        (data['x'] <= x_max) &
        (data['y'] >= y_min) &
        (data['y'] <= y_max) &
        (data['t'] >= t_min) &
        (data['t'] <= t_max)
    )

    return {
        k: v[mask]
        for k, v in data.items()
    }


# =========================================================
# 2. 空间分组
# =========================================================
def group_by_x(
        roi_data,
        group_size=200):

    x_min = roi_data['x'].min()
    x_max = roi_data['x'].max()

    bounds = np.arange(
        x_min,
        x_max + group_size,
        group_size
    )

    groups = []

    for i in range(len(bounds) - 1):

        x0 = bounds[i]
        x1 = bounds[i + 1]

        mask = (
            (roi_data['x'] >= x0) &
            (roi_data['x'] < x1)
        )

        if np.sum(mask) < 100:
            continue

        groups.append({
            'id': i,
            'x_range': (x0, x1),
            'data': {
                k: v[mask]
                for k, v in roi_data.items()
            }
        })

    return groups


# =========================================================
# 3. 3DFIT 拟合
# =========================================================
def fit_group_implicit(group):

    data = group['data']

    x_raw = data['x']
    y_raw = data['y']
    t_raw = data['t']

    # -----------------------------------------------------
    # 去中心化
    # -----------------------------------------------------
    x_m = np.mean(x_raw)
    y_m = np.mean(y_raw)
    t_0 = t_raw.min()

    x = x_raw - x_m
    y = y_raw - y_m
    t = t_raw - t_0

    # -----------------------------------------------------
    # 初始频率估计
    # -----------------------------------------------------
    y_c = y - np.mean(y)

    freqs = np.linspace(1, 300, 200)

    power = [
        np.abs(
            np.sum(
                y_c *
                np.exp(-1j * 2 * np.pi * f * t)
            )
        )
        for f in freqs
    ]

    f_init = freqs[np.argmax(power)]

    # -----------------------------------------------------
    # 初始参数
    # -----------------------------------------------------
    p0 = [
        0.0,
        1.0,
        np.ptp(y) / 2,
        f_init,
        0.0,
        0.1,
        0.0
    ]

    # -----------------------------------------------------
    # 目标函数
    # -----------------------------------------------------
    def objective(p):

        A, B, C, f, phi, lamb, D = p

        # 几何约束
        penalty = (
            1e6 *
            (np.sqrt(A**2 + B**2) - 1)**2
        )

        vibration = (
            C *
            np.exp(-lamb * t) *
            np.sin(
                2 * np.pi * f * t + phi
            )
        )

        residual = (
            A * x +
            B * y +
            vibration +
            D
        )

        return (
            np.mean(residual**2) +
            penalty
        )

    # -----------------------------------------------------
    # 优化
    # -----------------------------------------------------
    res = minimize(
        objective,
        p0,
        method='L-BFGS-B',
        bounds=[
            (-1, 1),
            (-1, 1),
            (0, None),
            (1, 500),
            (-np.pi, np.pi),
            (0, 10),
            (None, None)
        ]
    )

    if not res.success:
        return None

    popt = res.x

    # -----------------------------------------------------
    # 归一化
    # -----------------------------------------------------
    norm = np.sqrt(
        popt[0]**2 +
        popt[1]**2
    )

    popt[[0,1,2,6]] /= norm

    return {
        'params': popt,
        'x_m': x_m,
        'y_m': y_m,
        't_0': t_0,
        'id': group['id'],
        'x_range': group['x_range']
    }


# =========================================================
# 4. 使用拟合参数重建信号
# =========================================================
def reconstruct_signal(
        fit_result,
        roi_data,
        dt=0.001):

    # -----------------------------------------------------
    # 读取参数
    # -----------------------------------------------------
    p = fit_result['params']

    A, B, C, f, phi, lamb, D = p

    # -----------------------------------------------------
    # 时间轴
    # 与ECT/Gabor统一
    # -----------------------------------------------------
    t_min = roi_data['t'].min()
    t_max = roi_data['t'].max()

    times = np.arange(
        t_min,
        t_max,
        dt
    )

    tl = times - fit_result['t_0']

    # -----------------------------------------------------
    # 使用拟合参数直接生成振动信号
    # =====================================================
    # 关键：
    # 这里的频率就是拟合出的 f
    # 不再重新估计
    # -----------------------------------------------------
    signal = (
        C *
        np.exp(-lamb * tl) *
        np.sin(
            2 * np.pi * f * tl + phi
        )
    )

    signal -= np.mean(signal)

    return times, signal, f, C


# =========================================================
# 5. 频谱可视化（ECT风格）
# =========================================================
def plot_3dfit_spectrum(
        times,
        signal,
        fitted_freq):

    # -----------------------------------------------------
    # 可视化参数
    # -----------------------------------------------------
    grid_alpha = 0.3
    label_fontsize = 20
    tick_fontsize = 18

    # -----------------------------------------------------
    # FFT
    # -----------------------------------------------------
    fs = 1.0 / np.mean(np.diff(times))

    N = len(signal)

    freqs = np.fft.rfftfreq(
        N,
        d=1/fs
    )

    spectrum = np.abs(
        np.fft.rfft(signal)
    )

    # -----------------------------------------------------
    # 归一化
    # -----------------------------------------------------
    spectrum /= (
        spectrum.max() + 1e-12
    )

    # -----------------------------------------------------
    # 不再使用 FFT peak
    # 而是：
    # 使用拟合频率对应的位置
    # -----------------------------------------------------
    idx = np.argmin(
        np.abs(freqs - fitted_freq)
    )

    peak_freq = freqs[idx]
    peak_amp = spectrum[idx]

    # =====================================================
    # 频谱图
    # =====================================================
    plt.figure(figsize=(6, 5))

    plt.plot(
        freqs,
        spectrum,
        lw=2
    )

    # -----------------------------------------------------
    # 红点：拟合频率
    # -----------------------------------------------------
    plt.scatter(
        peak_freq,
        peak_amp,
        color='r',
        s=60,
        zorder=3,
        label=f"{fitted_freq:.2f} Hz"
    )

    plt.xlabel(
        "Frequency (Hz)",
        fontsize=label_fontsize
    )

    plt.ylabel(
        "Normalized Amplitude",
        fontsize=label_fontsize
    )

    plt.xlim(0, 250)

    plt.legend(
        frameon=False,
        fontsize=tick_fontsize,
        loc='upper right'
    )

    plt.grid(alpha=grid_alpha)

    plt.tick_params(
        axis='both',
        labelsize=tick_fontsize
    )

    plt.tight_layout()

    plt.show()


# =========================================================
# 6. main
# =========================================================
def main():

    # -----------------------------------------------------
    # 数据
    # -----------------------------------------------------
    file_path = "./data/30hz-20.hdf5"

    data = load_events(file_path)

    roi = select_large_roi(
        data,
        100, 600,
        420, 460,
        0, 1.0
    )

    print(
        f"Events in ROI: {len(roi['x'])}"
    )

    # -----------------------------------------------------
    # 分组
    # -----------------------------------------------------
    groups = group_by_x(
        roi,
        group_size=800
    )

    fitted = []

    # -----------------------------------------------------
    # 拟合
    # -----------------------------------------------------
    for g in groups:

        res = fit_group_implicit(g)

        if res:

            fitted.append(res)

            print(
                f"Group {res['id']} "
                f"Fitted: "
                f"f={res['params'][3]:.2f}Hz, "
                f"λ={res['params'][5]:.2f}"
            )

    if not fitted:

        print("拟合失败")
        return

    # =====================================================
    # 使用第一个group
    # =====================================================
    fit_result = fitted[0]

    # -----------------------------------------------------
    # 重建信号
    # -----------------------------------------------------
    times, signal, fitted_freq, amplitude = \
        reconstruct_signal(
            fit_result,
            roi,
            dt=0.001
        )

    # -----------------------------------------------------
    # 频谱图
    # -----------------------------------------------------
    plot_3dfit_spectrum(
        times,
        signal,
        fitted_freq
    )

    # =====================================================
    # 输出
    # =====================================================
    print("\n========== 3DFIT Result ==========")

    print(
        f"Fitted Frequency : "
        f"{fitted_freq:.3f} Hz"
    )

    print(
        f"Estimated Amplitude : "
        f"{amplitude:.3f} px"
    )


# =========================================================
# main
# =========================================================
if __name__ == "__main__":

    main()